In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# Define paths
ARTIFACT_DIR = Path('./artifacts')
UPSTREAM_MAP = ARTIFACT_DIR / 'upstream_map.json'
DOWNSTREAM_MAP = ARTIFACT_DIR / 'downstream_map.json'
DETECTED_ANOMALIES = ARTIFACT_DIR / 'detected_anomalies.json'
GRANGER_EDGES = ARTIFACT_DIR / 'causal_candidates_granger.csv'
ROOT_CAUSE_OUT = ARTIFACT_DIR / 'root_cause_candidates.csv'

# Traversal parameters
MAX_DEPTH = 3
DECAY_FACTOR = 0.8
SELF_ANOMALY_BONUS = 2.0  # Bonus if the candidate itself is anomalous

## Step 1: Load Frozen Causal Graph

In [ ]:
# Load downstream map (for correct RCA scoring)
if not DOWNSTREAM_MAP.exists():
    raise FileNotFoundError(f"Run Causal Discovery first to generate: {DOWNSTREAM_MAP}")

with open(DOWNSTREAM_MAP, 'r') as f:
    downstream_map = json.load(f)

print(f"Loaded downstream map: {len(downstream_map)} nodes have children")

## Step 2: Load Detected Anomalies

In [ ]:
# Load detected anomalies
if not DETECTED_ANOMALIES.exists():
    raise FileNotFoundError(f"Run Notebook 2 first to generate: {DETECTED_ANOMALIES}")

with open(DETECTED_ANOMALIES, 'r') as f:
    detected_anomalies = json.load(f)

print(f"Loaded {len(detected_anomalies)} detected anomalies")

# Display anomalies
print("\nDetected anomalies:")
for metric, details in list(detected_anomalies.items())[:5]:
    z = details.get('z_score')
    print(f"  {metric}: z={z:.2f if z else 0:.2f}")

## Step 3: Load Edge Weights (Optional)

In [ ]:
# Load edge weights from Granger results (if available)
edge_weights = {}

if GRANGER_EDGES.exists():
    granger_df = pd.read_csv(GRANGER_EDGES)
    
    for _, row in granger_df.iterrows():
        parent = row.get('from')
        child = row.get('to')
        min_p = row.get('min_p', 1.0)
        
        # Convert p-value to weight: smaller p-value = stronger edge
        # weight = -log10(p) capped at 10
        weight = min(-np.log10(max(min_p, 1e-10)), 10.0)
        edge_weights[(parent, child)] = weight
    
    print(f"\nLoaded edge weights for {len(edge_weights)} edges")
else:
    print("\nNo edge weights found — using uniform weights")

## Step 4: Score Root Cause Candidates

**Scoring Logic**: For RCA, we need to find metrics that CAUSE many downstream anomalies:
- Traverse **DOWNSTREAM** from each metric to count how many anomalies it can reach
- Metrics that are upstream of many anomalies get higher scores
- Add bonus if the metric itself is anomalous
- This naturally favors root causes without hardcoding assumptions

In [ ]:
def score_root_cause_candidates(
    anomalies,
    downstream_map,
    edge_weights=None,
    max_depth=3,
    decay=0.8,
    self_anomaly_bonus=2.0
):
    """
    Score root cause candidates by counting downstream anomalies they can reach.
    
    The intuition: A true root cause will have many anomalous metrics downstream.
    
    Args:
        anomalies: Dict of detected anomalies {metric: info}
        downstream_map: Dict mapping node -> list of child nodes
        edge_weights: Dict mapping (parent, child) -> weight
        max_depth: Maximum traversal depth
        decay: Score decay factor per hop
        self_anomaly_bonus: Bonus multiplier if candidate itself is anomalous
        
    Returns:
        Dict mapping candidate_metric -> score
    """
    anomalous_metrics = set(anomalies.keys())
    
    # Get all metrics that could be candidates (anomalous + their ancestors)
    all_candidates = set(anomalous_metrics)
    
    # Add all metrics that have any descendants (potential root causes)
    for metric in downstream_map.keys():
        all_candidates.add(metric)
    
    candidate_scores = {}
    
    for candidate in all_candidates:
        # BFS traversal downstream from this candidate
        visited = set()
        queue = [(candidate, 0, 1.0)]
        reachable_anomalies = set()
        total_score = 0.0
        
        while queue:
            current_node, depth, score = queue.pop(0)
            
            if current_node in visited or depth > max_depth:
                continue
            
            visited.add(current_node)
            
            # If this node is anomalous, count it
            if current_node in anomalous_metrics:
                reachable_anomalies.add(current_node)
                total_score += score
            
            # Expand to children
            if depth < max_depth:
                children = downstream_map.get(current_node, [])
                
                for child in children:
                    # Get edge weight
                    edge_weight = 1.0
                    if edge_weights:
                        edge_weight = edge_weights.get((current_node, child), 1.0)
                    
                    # Compute propagated score
                    propagated_score = score * decay * edge_weight
                    queue.append((child, depth + 1, propagated_score))
        
        # Score is the sum of reachable anomalies (with decay)
        candidate_scores[candidate] = total_score
        
        # Bonus if the candidate itself is anomalous
        if candidate in anomalous_metrics:
            candidate_scores[candidate] *= self_anomaly_bonus
    
    return candidate_scores

# Run scoring
candidate_scores = score_root_cause_candidates(
    detected_anomalies,
    downstream_map,
    edge_weights=edge_weights,
    max_depth=MAX_DEPTH,
    decay=DECAY_FACTOR,
    self_anomaly_bonus=SELF_ANOMALY_BONUS
)

print(f"\n{'='*60}")
print(f"Scored {len(candidate_scores)} root cause candidates")
print(f"{'='*60}")

## Step 5: Rank & Export Candidates

In [ ]:
# Convert to DataFrame and rank
candidates_df = pd.DataFrame([
    {'metric': metric, 'score': score}
    for metric, score in candidate_scores.items()
])

# Sort by score (descending)
candidates_df = candidates_df.sort_values('score', ascending=False).reset_index(drop=True)
candidates_df['rank'] = candidates_df.index + 1

print("\nTop-10 Root Cause Candidates:")
print(candidates_df.head(10).to_string(index=False))

# Save results
candidates_df.to_csv(ROOT_CAUSE_OUT, index=False)
print(f"\n✓ Saved root cause candidates to: {ROOT_CAUSE_OUT}")

print("\n" + "="*60)
print("✓ Notebook 3 Complete — Root Causes Ranked")
print("="*60)